# Time Series Graphics — Student Version

Run all cells from top to bottom in Google Colab.


## What we are learning

Before forecasting, we must understand the history of the data. We will learn to:

1. represent time-series data in a pandas DataFrame;
2. create time plots, seasonal plots, and seasonal subseries plots;
3. recognize trend, seasonality, cycles, and unusual observations;
4. use scatterplots and correlation;
5. use lag plots and autocorrelation; and
6. recognize white noise.

**Graphs are evidence for choosing a forecasting method.**


## 1. Setup

Run the next two cells first. They import the libraries and download the data.

In [ ]:
# Import the libraries used in the lesson.
# pandas = tables; numpy = numerical work; matplotlib and seaborn = charts.
# statsmodels = autocorrelation charts.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

# Reproducibility and readable charts.
np.random.seed(1)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 3)

# Colab sessions are temporary, so create a folder for the downloaded data.
DATA_DIR = Path('/content/fpppy_data')
DATA_DIR.mkdir(exist_ok=True)
print('Setup complete.')


In [ ]:
# Download stable public copies of the datasets used in this lesson.
# Files are saved locally in Colab so every later cell can run from the beginning.
DATA_URLS = {
    'ansett.csv': 'https://raw.githubusercontent.com/zgana/fpp3-python-readalong/master/data/tsibbledata/ansett.csv',
    'aus_production.csv': 'https://raw.githubusercontent.com/zgana/fpp3-python-readalong/master/data/tsibbledata/aus_production.csv',
    'PBS.csv': 'https://raw.githubusercontent.com/zgana/fpp3-python-readalong/master/data/tsibbledata/PBS.csv',
    'vic_elec.csv': 'https://raw.githubusercontent.com/zgana/fpp3-python-readalong/master/data/tsibbledata/vic_elec.csv',
}
for filename, url in DATA_URLS.items():
    destination = DATA_DIR / filename
    pd.read_csv(url).to_csv(destination, index=False)
    print('Ready:', filename)


## 2. DataFrames and dates

A DataFrame is a table. A time series needs measurements and the times at which they were observed.

In [ ]:
simple = pd.DataFrame({
    'Year': [2015, 2016, 2017, 2018, 2019],
    'Observation': [123, 39, 78, 52, 110],
})
display(simple)
print(simple.dtypes)
print(pd.to_datetime(['2020-01-01', '2020-01-02', '2020-01-03']))


## 3. Time plot: airline passengers

In [ ]:
ansett = pd.read_csv(DATA_DIR / 'ansett.csv')
ansett['Week'] = pd.to_datetime(
    ansett['Week'].str.replace(' ', '-') + '-1', format='%G-W%V-%u'
)
melsyd_economy = ansett[(ansett['Airports'] == 'MEL-SYD') & (ansett['Class'] == 'Economy')].copy()
melsyd_economy['Passengers_thousands'] = melsyd_economy['Passengers'] / 1000

fig, ax = plt.subplots()
ax.plot(melsyd_economy['Week'], melsyd_economy['Passengers_thousands'])
ax.set(title='Ansett economy passengers: Melbourne–Sydney', xlabel='Week', ylabel='Passengers (thousands)')
fig.autofmt_xdate()
plt.show()


## 4. Trend, seasonality, and cycles

Trend is long-term movement. Seasonality repeats with a fixed calendar period. Cyclicity is a longer, irregular rise and fall.

In [ ]:
pbs = pd.read_csv(DATA_DIR / 'PBS.csv')
pbs['Month'] = pd.to_datetime(pbs['Month'])
total_cost = pbs[pbs['ATC2'] == 'A10'].groupby('Month', as_index=False)['Cost'].sum()
total_cost['Cost_millions'] = total_cost['Cost'] / 1_000_000

fig, ax = plt.subplots()
ax.plot(total_cost['Month'], total_cost['Cost_millions'])
ax.set(title='Australian antidiabetic drug sales', xlabel='Month', ylabel='$ millions')
fig.autofmt_xdate()
plt.show()


## 5. Seasonal plot

In [ ]:
seasonal_df = total_cost.assign(
    Year=total_cost['Month'].dt.year,
    Month_number=total_cost['Month'].dt.month,
)
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=seasonal_df, x='Month_number', y='Cost_millions', hue='Year', marker='o', palette='viridis', ax=ax)
ax.set(title='Seasonal plot: antidiabetic drug sales', xlabel='Month', ylabel='$ millions', xticks=range(1, 13))
ax.legend(title='Year', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()


## 6. Seasonal subseries plot

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8), sharey=True)
for month_number, month_df in seasonal_df.groupby('Month_number'):
    ax = axes.flat[month_number - 1]
    ax.plot(month_df['Year'], month_df['Cost_millions'], marker='o', color='black')
    ax.axhline(month_df['Cost_millions'].mean(), color='tab:blue')
    ax.set_title(pd.Timestamp(2000, month_number, 1).strftime('%b'))
    ax.tick_params(axis='x', rotation=45)
fig.suptitle('Seasonal subseries: antidiabetic drug sales')
fig.supxlabel('Year')
fig.supylabel('$ millions')
plt.tight_layout()
plt.show()


## 7. Scatterplot and correlation

In [ ]:
vic = pd.read_csv(DATA_DIR / 'vic_elec.csv')
vic['Time'] = pd.to_datetime(vic['Time'])
vic_2014 = vic[vic['Time'].dt.year == 2014].copy()

fig, ax = plt.subplots()
sns.scatterplot(data=vic_2014, x='Temperature', y='Demand', s=12, alpha=0.35, ax=ax)
ax.set(title='Electricity demand versus temperature', xlabel='Temperature (°C)', ylabel='Demand')
plt.show()
print(vic_2014[['Temperature', 'Demand']].corr())


## 8. Lag plots

In [ ]:
production = pd.read_csv(DATA_DIR / 'aus_production.csv')
production['Quarter'] = pd.PeriodIndex(production['Quarter'].str.replace(' ', ''), freq='Q').to_timestamp()
beer = production.loc[production['Quarter'] >= '2000', ['Quarter', 'Beer']].dropna().copy()

fig, axes = plt.subplots(3, 3, figsize=(11, 10), sharex=True, sharey=True)
for lag, ax in enumerate(axes.flat, start=1):
    lagged = beer.assign(Past_value=beer['Beer'].shift(lag)).dropna()
    sns.scatterplot(data=lagged, x='Past_value', y='Beer', ax=ax, s=45)
    ax.set_title(f'Lag {lag}')
plt.tight_layout()
plt.show()


## 9. Autocorrelation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(beer['Beer'], lags=16, zero=False, ax=ax)
ax.set(title='ACF: quarterly beer production', xlabel='Lag (quarters)', ylabel='Autocorrelation')
plt.show()


## 10. White noise

In [ ]:
rng = np.random.default_rng(30)
white_noise = rng.normal(0, 1, 50)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(white_noise, marker='o')
axes[0].set(title='White noise', xlabel='Observation', ylabel='Value')
plot_acf(white_noise, lags=16, zero=False, ax=axes[1])
axes[1].set(title='ACF of white noise', xlabel='Lag', ylabel='Autocorrelation')
plt.tight_layout()
plt.show()


## Practice questions

Choose one series and answer:

1. What is its observation frequency?
2. Do you see trend, seasonality, cycles, or unusual values?
3. Which graph helped you most, and why?
4. What does the ACF suggest?
5. What should a forecasting method capture?